In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Paglalarawan ng mga quantum computer para sa transpiler


<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Para ma-convert ang isang abstract circuit sa isang ISA circuit na maaaring tumakbo sa isang partikular na QPU (quantum processing unit), kailangan ng transpiler ng ilang impormasyon tungkol sa QPU. Ang impormasyong ito ay makikita sa dalawang lugar: ang `BackendV2` (o legacy na `BackendV1`) na object na plano mong gamitin para mag-submit ng mga job, at ang `Target` attribute ng backend.

- Ang [`Target`](../api/qiskit/qiskit.transpiler.Target) ay naglalaman ng lahat ng nauugnay na mga limitasyon ng isang device, tulad ng mga native basis gate nito, koneksyon ng qubit, at impormasyon tungkol sa pulse o timing.
- Ang [`Backend`](../api/qiskit/qiskit.providers.BackendV2) ay may kasamang `Target` bilang default, naglalaman ng karagdagang impormasyon -- tulad ng [`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap), at nagbibigay ng interface para sa pag-submit ng mga quantum circuit job.

Maaari ka ring explicitly na magbigay ng impormasyon para gamitin ng transpiler, halimbawa, kung mayroon kang tiyak na use case, o kung naniniwala kang makakatulong ang impormasyong ito sa transpiler na makabuo ng mas optimized na circuit.

Ang katumpakan ng transpiler sa paglikha ng pinakaangkop na circuit para sa partikular na hardware ay nakasalalay sa dami ng impormasyon na mayroon ang `Target` o `Backend` tungkol sa mga limitasyon nito.

> **Note:** Dahil marami sa mga pinagbabatayan na transpilation algorithm ay stochastic, walang garantiya na mahahanap ang mas magandang circuit.

Ipinapakita ng pahinang ito ang ilang halimbawa ng pagpapasa ng impormasyon ng QPU sa transpiler. Ginagamit ng mga halimbawang ito ang target mula sa [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) mock backend.

<span id="default-config"></span>
## Default na configuration
Ang pinakasimpleng paggamit ng transpiler ay ang magbigay ng lahat ng impormasyon ng QPU sa pamamagitan ng pagbibigay ng `Backend` o `Target`. Para mas maunawaan kung paano gumagana ang transpiler, bumuo ng circuit at i-transpile ito gamit ang iba't ibang impormasyon, tulad ng sumusunod.

I-import ang mga kinakailangang library at i-instantiate ang QPU:
Para ma-convert ang isang abstract circuit sa isang ISA circuit na maaaring tumakbo sa isang partikular na processor, kailangan ng transpiler ng ilang impormasyon tungkol sa processor. Karaniwan, ang impormasyong ito ay naka-store sa [`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) o [`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target) na ibinibigay sa transpiler, at hindi na kailangan ng karagdagang impormasyon. Gayunpaman, maaari ka ring explicitly na magbigay ng impormasyon para gamitin ng transpiler, halimbawa, kung mayroon kang tiyak na use case, o kung naniniwala kang makakatulong ang impormasyong ito sa transpiler na makabuo ng mas optimized na circuit.

Ipinapakita ng paksang ito ang ilang halimbawa ng pagpapasa ng impormasyon sa transpiler. Ginagamit ng mga halimbawang ito ang target mula sa [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) mock backend.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

Ang halimbawang circuit ay gumagamit ng instance ng [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) mula sa circuit library ng Qiskit.

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

Ginagamit ng halimbawang ito ang mga default na setting para i-transpile sa `target` ng `backend`, na nagbibigay ng lahat ng impormasyong kailangan para ma-convert ang circuit sa isang maaaring tumakbo sa backend.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

Ginagamit ang halimbawang ito sa mga susunod na seksyon ng paksang ito para ipakita na ang coupling map at basis gates ang mahahalagang piraso ng impormasyon na ipapasa sa transpiler para sa pinakamainam na pagbuo ng circuit. Karaniwan, maaaring pumili ang QPU ng mga default na setting para sa iba pang impormasyon na hindi ipinasa, tulad ng timing at scheduling.

## Coupling map

Ang coupling map ay isang graph na nagpapakita kung aling mga qubit ang magkakaugnay at samakatuwid ay may two-qubit gates sa pagitan nila. Minsan ang graph na ito ay may direksyon, ibig sabihin ang mga two-qubit gate ay isang direksyon lang maaaring tumakbo. Gayunpaman, palagi naming mababago ng transpiler ang direksyon ng gate sa pamamagitan ng pagdaragdag ng mga karagdagang single-qubit gate. Ang isang abstract quantum circuit ay palaging maaaring katawanin sa graph na ito, kahit limitado ang koneksyon nito, sa pamamagitan ng pagdaragdag ng mga SWAP gate para ilipat ang quantum na impormasyon.

Ang mga qubit mula sa ating mga abstract circuit ay tinatawag na _virtual qubits_ at ang mga nasa coupling map ay _physical qubits_. Nagbibigay ang transpiler ng mapping sa pagitan ng virtual at physical qubits. Isa sa mga unang hakbang sa transpilation, ang _layout_ stage, ang gumaganap ng mapping na ito.

> **Note:** Kahit magkaugnay ang routing stage at ang _layout_ stage — na pumipili ng aktwal na mga qubit — bilang default, itinuturing ng paksang ito ang mga ito bilang magkahiwalay na mga stage para sa pagiging simple. Ang kumbinasyon ng routing at layout ay tinatawag na _qubit mapping_. Matuto pa tungkol sa mga stage na ito sa paksa ng [Mga stage ng Transpiler](transpiler-stages).

Ipasa ang `coupling_map` keyword argument para makita ang epekto nito sa transpiler:

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

Tulad ng nakikita sa itaas, ilang SWAP gates ang naipasok (bawat isa ay binubuo ng tatlong CX gates), na magdudulot ng maraming error sa mga kasalukuyang device. Para makita kung aling mga qubit ang napili sa aktwal na qubit topology, gamitin ang `plot_circuit_layout` mula sa Qiskit Visualizations:

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

Ipinapakita nito na ang ating mga virtual qubit 0-11 ay simpleng na-map sa linya ng mga physical qubit 0-11. Bumalik tayo sa default (`optimization_level=1`), na gumagamit ng `VF2Layout` kung kailangan ng anumang routing.

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

Ngayon walang mga SWAP gate na naipasok at ang mga physical qubit na napili ay katulad noong ginamit ang `target` class.

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

Ngayon ang layout ay nasa isang ring. Dahil iginagalang ng layout na ito ang koneksyon ng circuit, walang mga SWAP gate, na nagbibigay ng mas magandang circuit para sa pagpapatupad.

## Basis gates

Bawat quantum computer ay sumusuporta sa limitadong set ng instruksyon, na tinatawag na _basis gates_ nito. Bawat gate sa circuit ay dapat i-translate sa mga elemento ng set na ito. Ang set na ito ay dapat binubuo ng mga single- at two-qubit gate na nagbibigay ng universal gates set, ibig sabihin maaaring i-decompose ang anumang quantum operation sa mga gate na iyon. Ginagawa ito ng [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator), at ang basis gates ay maaaring tukuyin bilang keyword argument sa transpiler para ibigay ang impormasyong ito.

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

Ang mga default na single-qubit gate sa _ibm_sherbrooke_ ay `rz`, `x`, at `sx`, at ang default na two-qubit gate ay `ecr` (echoed cross-resonance). Ang mga CX gate ay ginawa mula sa mga `ecr` gate, kaya sa ilang QPU ang `ecr` ang tinukoy bilang two-qubit basis gate, habang sa iba ang `cx` ang default. Ang `ecr` gate ay ang _entangling_ na bahagi ng `cx` gate. Bukod sa mga control gate, mayroon ding mga `delay` at `measurement` na instruksyon.

> **Note:** Ang mga QPU ay may mga default na basis gate, ngunit maaari kang pumili ng anumang gate na gusto mo, basta't nagbigay ka ng instruksyon o nagdaragdag ng pulse gates (tingnan ang [Gumawa ng mga transpiler pass.](custom-transpiler-pass)) Ang mga default na basis gate ay ang mga nagkaroon ng calibration sa QPU, kaya hindi na kailangang magbigay ng karagdagang instruksyon/pulse gate. Halimbawa, sa ilang QPU ang `cx` ang default na two-qubit gate at `ecr` naman sa iba. Tingnan ang listahan ng posibleng [native gates at operations](/guides/qpu-information#native-gates) para sa karagdagang detalye.

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Output ng nakaraang code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

Pansinin na ang mga `CXGate` object ay na-decompose sa mga `ecr` gate at single-qubit basis gate.
## Mga rate ng error ng device
Ang `Target` class ay maaaring maglaman ng impormasyon tungkol sa mga rate ng error para sa mga operasyon sa device.
Halimbawa, kinukuha ng sumusunod na code ang mga katangian para sa echoed cross-resonance (ECR) gate sa pagitan ng qubit 1 at 0 (pansinin na ang ECR gate ay may direksyon):

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

Ipinapakita ng output ang tagal ng gate (sa segundo) at ang rate ng error nito. Para ipakita ang impormasyon ng error sa transpiler, bumuo ng target model gamit ang `basis_gates` at `coupling_map` mula sa itaas at punan ito ng mga halaga ng error mula sa `FakeSherbrooke` backend.